# CIFAR-100 Training Notebook (Colab)

Train FracBNN / Ada-FracBNN on CIFAR-100 (100 classes, 32x32 images).

| model\_id | Architecture | Description |
|-----------|-------------|-------------|
| 0 | `binput-pg` | Baseline PG |
| 1 | `adaptive-pg` | Adaptive PG (Ada-FracBNN) |
| 2 | `adaptive-pg-kd` | Adaptive PG + Knowledge Distillation |

## Before you start
1. **Enable GPU**: Runtime > Change runtime type > **T4 GPU**.
2. **Mount Google Drive** (optional): for saving checkpoints across sessions.

### Dataset
CIFAR-100 is downloaded automatically by torchvision -- no manual data setup needed.
50K training images, 10K test images, 100 classes, 32x32 resolution.

### Colab paths
- `/content/` -- writable working directory
- Session limit: ~12 h (free tier), longer with Pro
- CIFAR-100 is small (~170 MB) and trains fast on T4

## Cell 1 -- GPU and Disk Check

In [ ]:
!nvidia-smi
import torch, os
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"VRAM: {mem_gb:.1f} GB")
    print(f"GPU count: {torch.cuda.device_count()}")
!df -h /content

## Cell 2 -- Clone Repo

In [ ]:
import os

REPO_DIR = "/content/endingengineering"
REPO_URL = "https://github.com/unworld11/endingengineering.git"
BRANCH   = "feat/imagenet-colab-bootstrap"

if not os.path.exists(REPO_DIR):
    !git clone --branch $BRANCH $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git fetch origin $BRANCH && git checkout $BRANCH && git pull origin $BRANCH

os.chdir(REPO_DIR)
!git log --oneline -5

## Cell 3 -- Install Dependencies

In [ ]:
!pip install -q -r requirements.txt

## Cell 4 -- Smoke Test (verify CIFAR models load)

In [ ]:
!python test_models.py

## Cell 5 -- Set Data & Checkpoint Directories

CIFAR-100 auto-downloads via torchvision. We just need writable directories.

In [ ]:
import os

DATA_DIR = "/content/cifar100_data"
SAVE_DIR = "/content/checkpoints"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"CIFAR-100 will be downloaded to {DATA_DIR}")
print(f"Checkpoints will be saved to {SAVE_DIR}")

## Cell 6 -- (Optional) Mount Google Drive for Persistent Checkpoints

Skip this cell if you don't need checkpoints to survive session resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/cifar100_checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Checkpoints will persist to Google Drive: {SAVE_DIR}")

---
## Cell 7 -- Train Model 0: Baseline PG (`binput-pg`)

CIFAR-100: 100 classes, 32x32 images. ~390 batches/epoch at batch 128. Very fast on T4.

In [ ]:
os.chdir(REPO_DIR)

!CUDA_VISIBLE_DEVICES=0 python cifar10.py \
    --dataset cifar100 \
    -id 0 \
    -e  250 \
    -b  128 \
    -lr 1e-3 \
    -wd 1e-5 \
    -d  $DATA_DIR \
    --label_smoothing 0.05 \
    -gpu 0 \
    -s

!cp -a save_CIFAR100_model "$SAVE_DIR/save_CIFAR100_model_baseline"

## Cell 8 -- Train Model 1: Adaptive PG (`adaptive-pg`) on CIFAR-100

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python cifar10.py \
    --dataset cifar100 \
    -id 1 \
    -e  250 \
    -b  128 \
    -lr 1e-3 \
    -wd 1e-5 \
    -d  $DATA_DIR \
    -ts 0.15 \
    -sw 0.01 \
    --label_smoothing 0.05 \
    -gpu 0 \
    -s

!cp -a save_CIFAR100_model "$SAVE_DIR/save_CIFAR100_model_adaptive"

## Cell 8.5 -- Train Teacher Model for KD

Full-precision ResNet-20 teacher on CIFAR-100. Takes ~1 hour on T4.
The saved checkpoint is used by Cell 9 (KD training) below.

In [ ]:
os.chdir(REPO_DIR)

!CUDA_VISIBLE_DEVICES=0 python train_teacher.py \
    --dataset cifar100 \
    -e  250 \
    -b  128 \
    -lr 0.1 \
    -wd 1e-4 \
    -d  $DATA_DIR \
    -gpu 0 \
    -s

TEACHER_CKPT = os.path.join(REPO_DIR, "save_teacher_model", "model_teacher.pt")
print(f"\nTeacher saved to: {TEACHER_CKPT}")
print(f"Exists: {os.path.exists(TEACHER_CKPT)}")

## Cell 9 -- Train Model 2: Adaptive PG + KD (`adaptive-pg-kd`) on CIFAR-100

Uses the teacher checkpoint from Cell 8.5 above. Run that cell first.

In [ ]:
TEACHER_PATH = os.path.join(REPO_DIR, "save_teacher_model", "model_teacher.pt")

if not os.path.exists(TEACHER_PATH):
    print(f"Teacher checkpoint not found at {TEACHER_PATH}")
    print("Run Cell 8.5 first to train the teacher.")
else:
    print(f"Using teacher: {TEACHER_PATH}")
    !CUDA_VISIBLE_DEVICES=0 python cifar10.py \
        --dataset cifar100 \
        -id 2 \
        -e  250 \
        -b  128 \
        -lr 1e-3 \
        -wd 1e-5 \
        -d  $DATA_DIR \
        -ts 0.15 \
        -sw 0.01 \
        -temp  4.0 \
        -alpha 0.7 \
        -tp $TEACHER_PATH \
        --label_smoothing 0.05 \
        -gpu 0 \
        -s

    !cp -a save_CIFAR100_model "$SAVE_DIR/save_CIFAR100_model_kd"

## Cell 10 -- Evaluate All Saved CIFAR-100 Models

In [ ]:
import glob

variants = [
    ("Baseline PG",      0, f"{SAVE_DIR}/save_CIFAR100_model_baseline"),
    ("Adaptive PG",      1, f"{SAVE_DIR}/save_CIFAR100_model_adaptive"),
    ("Adaptive PG + KD", 2, f"{SAVE_DIR}/save_CIFAR100_model_kd"),
]

for name, mid, path in variants:
    ckpts = sorted(glob.glob(os.path.join(path, "model_*.pt")))
    if not ckpts:
        print(f"\n[SKIP] {name} -- no checkpoint found in {path}")
        continue
    ckpt = ckpts[0]
    print(f"\n{'='*60}")
    print(f"Evaluating: {name}  (checkpoint: {ckpt})")
    print(f"{'='*60}")
    !CUDA_VISIBLE_DEVICES=0 python cifar10.py --dataset cifar100 -id $mid -t -r "$ckpt" -d $DATA_DIR -gpu 0

## Cell 11 -- Resume Training After Session Timeout

Re-run Cells 1-5 (fast -- repo is cached), then use this cell.  
Edit the variables below to match the variant you were training.

In [ ]:
# --- edit these lines to match your variant ---
MODEL_ID   = 1
CKPT_MODEL = f"{SAVE_DIR}/save_CIFAR100_model_adaptive/model_adaptive-pg.pt"
# -----------------------------------------------

os.chdir(REPO_DIR)

!CUDA_VISIBLE_DEVICES=0 python cifar10.py \
    --dataset cifar100 \
    -id $MODEL_ID \
    -e  250 \
    -b  128 \
    -lr 1e-3 \
    -wd 1e-5 \
    -r  "$CKPT_MODEL" \
    -d  $DATA_DIR \
    -ts 0.15 \
    -sw 0.01 \
    --label_smoothing 0.05 \
    -gpu 0 \
    -s

---
## Reference

### Hyperparameters for CIFAR-100

| Parameter | Value | Notes |
|-----------|-------|-------|
| Epochs | 250 | Cosine schedule with 5 warmup epochs |
| GPU | 1 | Single T4 is plenty for 32x32 images |
| Batch size | 128 | |
| Learning rate | 1e-3, cosine decay | |
| Weight decay | 1e-5 | |
| Label smoothing | 0.05 | |
| Mixup | 0.2 (adaptive models) | Auto-enabled for adaptive-pg variants |
| Target sparsity | 0.15 | 15% channels get 2-bit |
| Sparsity weight | 0.01 | |
| KD temperature | 4.0 | |
| KD alpha | 0.7 | |

### Expected training time

- ~390 batches/epoch at batch 128
- ~10-15 seconds per epoch on T4
- 250 epochs in ~1-1.5 hours
- All 3 variants finish well within one 12h session

### Colab runtime notes

- Free tier: T4 GPU, ~12 h session limit
- CIFAR-100 is ~170 MB and auto-downloads on first run
- Mount Google Drive (Cell 6) to persist checkpoints across sessions